In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
import sys
from torch.utils.data import Dataset, DataLoader, RandomSampler
import math
from collections import OrderedDict
from torch.optim import AdamW

In [2]:
#!pip install torchinfo
from torchinfo import summary

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
# Set the environment variable TOKENIZERS_PARALLELISM to 'false'
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [5]:
#!pip install transformers==4.49.0
#!pip install datasets

In [6]:
from transformers import AutoTokenizer

In [7]:
torch.__version__

'2.2.2'

In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len, device):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model, device=device)
        position = torch.arange(0., max_len,
                                device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0., d_model, 2, device=device) * -(math.log(10000.0) / d_model))
        pe_pos = torch.mul(position, div_term)
        pe[:, 0::2] = torch.sin(pe_pos)
        pe[:, 1::2] = torch.cos(pe_pos)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        out = self.pe[:, :x.size(1)].requires_grad_(False)
        return out

In [9]:
#del pe
pe_e = PositionalEncoding(d_model=768, dropout=0.1, max_len=512, device=device)
inp_tok = torch.tensor([5,8,78, 86, 78, 90, 45]).unsqueeze(0)
inp_tok, inp_tok.shape

(tensor([[ 5,  8, 78, 86, 78, 90, 45]]), torch.Size([1, 7]))

In [10]:
pe_e_e = pe_e(inp_tok)
pe_e_e, pe_e_e.shape

(tensor([[[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
            0.0000e+00,  1.0000e+00],
          [ 8.4147e-01,  5.4030e-01,  8.2843e-01,  ...,  1.0000e+00,
            1.0243e-04,  1.0000e+00],
          [ 9.0930e-01, -4.1615e-01,  9.2799e-01,  ...,  1.0000e+00,
            2.0486e-04,  1.0000e+00],
          ...,
          [-7.5680e-01, -6.5364e-01, -6.9153e-01,  ...,  1.0000e+00,
            4.0971e-04,  1.0000e+00],
          [-9.5892e-01,  2.8366e-01, -9.8573e-01,  ...,  1.0000e+00,
            5.1214e-04,  1.0000e+00],
          [-2.7942e-01,  9.6017e-01, -4.1267e-01,  ...,  1.0000e+00,
            6.1457e-04,  1.0000e+00]]], device='cuda:0'),
 torch.Size([1, 7, 768]))

In [11]:
class Embed(nn.Module):
    def __init__(self, vocab_size, embed_dim, ctx_len, do, device):
        super().__init__()
        self.tok_layer = nn.Embedding(vocab_size, embed_dim)
        self.pos_layer = PositionalEncoding(embed_dim, do, ctx_len, device)
        self.norm_do = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(do)
            )
        self.device = device
        self.ctx_len = ctx_len

    def forward(self, inp):
        inp = inp.to(self.device)
        len_inp = inp.shape[-1]
        try:
            assert len_inp <= self.ctx_len
        except:
            print("Err..Errr. Error...Bro..Length of supplied text exceeds context length defined.Exiting the program now")
            sys.exit(1)
        tok_embed = self.tok_layer(inp)
        pos_embed = self.pos_layer(inp)
        embed_tok_pos = tok_embed + pos_embed
        embed_out = self.norm_do(embed_tok_pos)
        return embed_out

In [12]:
inp_tok = torch.tensor([5,8,78, 86, 78, 90, 45]).unsqueeze(0)
inp_tok, inp_tok.shape

(tensor([[ 5,  8, 78, 86, 78, 90, 45]]), torch.Size([1, 7]))

In [13]:
emb = Embed(100, 768, 512, 0.1, device).to(device)

In [14]:
emb_out = emb(inp_tok.to(device))
emb_out.shape, emb_out

(torch.Size([1, 7, 768]),
 tensor([[[-1.8676, -0.1902, -0.1381,  ..., -0.3382,  0.8241,  0.0000],
          [-0.3042,  0.1276,  0.1784,  ...,  0.7371, -1.7845,  1.3150],
          [-0.0380, -0.9801,  0.0000,  ...,  0.9940, -1.9360,  0.6408],
          ...,
          [-1.6321, -1.1280, -0.4680,  ...,  1.0271, -1.8211,  0.6838],
          [-0.7497, -1.8066, -2.1681,  ...,  0.9868, -0.2734,  0.0000],
          [-0.2341,  1.9227, -0.1093,  ...,  1.7198,  0.5320,  1.1409]]],
        device='cuda:0', grad_fn=<NativeDropoutBackward0>))

In [15]:
summary(emb)

Layer (type:depth-idx)                   Param #
Embed                                    --
├─Embedding: 1-1                         76,800
├─PositionalEncoding: 1-2                --
│    └─Dropout: 2-1                      --
├─Sequential: 1-3                        --
│    └─LayerNorm: 2-2                    1,536
│    └─Dropout: 2-3                      --
Total params: 78,336
Trainable params: 78,336
Non-trainable params: 0

In [16]:
def att_mask(attention_mask, lookahead, cross_att, x=None):
    if cross_att:
        batch_dim = x[0]
        repeat = x[1]
    else:
        batch_dim = attention_mask.shape[0]
        repeat = len(attention_mask[0])

    mask =[]
    for i in range(batch_dim):
        am_interim = [attention_mask[i].tolist()] * repeat
        am_interim = torch.tensor(am_interim).unsqueeze(0)
        mask.append(am_interim)
    mask = torch.vstack(mask)
    if lookahead:
        inp_save = mask
        mask = torch.tril(torch.ones(mask.shape))
    mask = torch.where(mask == 0, -torch.inf, 0.0)
    return mask

In [17]:
attention_mask = torch.tensor([[1,1,1,1,0,0], [1,1,1,0,0,0]])
attention_mask.shape

torch.Size([2, 6])

In [18]:
mask  = att_mask(attention_mask, lookahead=False, cross_att=False, x=[2,9,6])

In [19]:
mask, mask.shape

(tensor([[[0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf]],
 
         [[0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf]]]),
 torch.Size([2, 6, 6]))

In [20]:
class Attention(nn.Module):
    def __init__(self, embed_dim, k_dim, do, device):
        super().__init__()
        self.embed_dim = embed_dim
        self.k_dim = k_dim
        self.query = nn.Linear(embed_dim, k_dim)
        self.key = nn.Linear(embed_dim, k_dim)
        self.value = nn.Linear(embed_dim, k_dim)
        self.att_do = nn.Dropout(do)
        self.device = device

    def forward(self, qry, ky, vlu, mask):
        q = self.query(qry)
        k = self.key(ky)
        v = self.value(vlu)
        qk = (q@k.transpose(1, 2))/(self.k_dim**0.5)
        mask = mask.to(self.device)
        qk_m = qk + mask
        qk_m_smax = torch.softmax(qk_m, dim=-1)
        qk_m_smax_do = self.att_do(qk_m_smax)
        qkv = qk_m_smax_do@v
        return qkv

In [21]:
class Attention_Block(nn.Module):
    def __init__(self, num_heads, embed_dim, k_dim, do, device):
        super().__init__()
        self.num_heads = num_heads
        self.heads_list = [Attention(embed_dim, k_dim, do, device) for i in range(num_heads)]
        self.heads = nn.ModuleList(self.heads_list)
        self.lin_do = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Dropout(do)
        )

    def forward(self, q, k, v, mask):
        heads_list_out = [head(q, k, v, mask) for head in self.heads]
        att_head = torch.cat(heads_list_out, dim=-1)
        att_head_out = self.lin_do(att_head)
        return att_head_out

In [22]:
class Encoder_Block(nn.Module):
    def __init__(self, num_heads, embed_dim, k_dim, do, device):
        super().__init__()
        self.MHA = Attention_Block(num_heads, embed_dim, k_dim, do, device)
        self.mha_blk_end_layernorm = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4),
            nn.GELU(),
            nn.Linear(embed_dim*4, embed_dim),
            nn.Dropout(do)
            )
        self.enc_blk_end_layernorm = nn.LayerNorm(embed_dim)

    def forward(self, args_list):  #q, k, v, mask):
        x = args_list[0]
        mask = args_list[1]

        inp_start_att_block = x
        x = self.MHA(x, x, x, mask)

        x = x + inp_start_att_block
        x = self.mha_blk_end_layernorm(x)

        inp_start_ff_block = x
        x = self.ff(x)

        x = x + inp_start_ff_block
        x = self.enc_blk_end_layernorm(x)
        return [x, mask]

In [23]:
class Encoder(nn.Module):
    def __init__(self, num_layers, num_heads, vocab_size, embed_dim, k_dim, ctx_len, do, device):
        super().__init__()
        self.emb = Embed(vocab_size, embed_dim, ctx_len, do, device)
        self.layer_list = [Encoder_Block(num_heads, embed_dim, k_dim, do, device) for i in range(num_layers)]
        self.layers = nn.Sequential(*self.layer_list)

    def forward(self, input_ids, attention_mask):
        x = self.emb(input_ids)
        mask = att_mask(attention_mask, lookahead=False, cross_att=False, x=None)
        x = self.layers([x, mask])
        return x[0]

In [29]:
enc = Encoder(3, 6, 100, 768, 128, 100, 0.1, device)
enc.to(device)

Encoder(
  (emb): Embed(
    (tok_layer): Embedding(100, 768)
    (pos_layer): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (norm_do): Sequential(
      (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (layers): Sequential(
    (0): Encoder_Block(
      (MHA): Attention_Block(
        (heads): ModuleList(
          (0-5): 6 x Attention(
            (query): Linear(in_features=768, out_features=128, bias=True)
            (key): Linear(in_features=768, out_features=128, bias=True)
            (value): Linear(in_features=768, out_features=128, bias=True)
            (att_do): Dropout(p=0.1, inplace=False)
          )
        )
        (lin_do): Sequential(
          (0): Linear(in_features=768, out_features=768, bias=True)
          (1): Dropout(p=0.1, inplace=False)
        )
      )
      (mha_blk_end_layernorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
  

In [31]:
summary(enc)

Layer (type:depth-idx)                        Param #
Encoder                                       --
├─Embed: 1-1                                  --
│    └─Embedding: 2-1                         76,800
│    └─PositionalEncoding: 2-2                --
│    │    └─Dropout: 3-1                      --
│    └─Sequential: 2-3                        --
│    │    └─LayerNorm: 3-2                    1,536
│    │    └─Dropout: 3-3                      --
├─Sequential: 1-2                             --
│    └─Encoder_Block: 2-4                     --
│    │    └─Attention_Block: 3-4              2,362,368
│    │    └─LayerNorm: 3-5                    1,536
│    │    └─Sequential: 3-6                   4,722,432
│    │    └─LayerNorm: 3-7                    1,536
│    └─Encoder_Block: 2-5                     --
│    │    └─Attention_Block: 3-8              2,362,368
│    │    └─LayerNorm: 3-9                    1,536
│    │    └─Sequential: 3-10                  4,722,432
│    │    └─LayerNor

In [32]:
inp = torch.randint(1,100, (4,6))
am = torch.ones(inp.shape)
inp.shape, am.shape

(torch.Size([4, 6]), torch.Size([4, 6]))

In [33]:
out = enc(inp.to(device), am.to(device))
out.shape

torch.Size([4, 6, 768])

In [34]:
from datasets import load_dataset

In [35]:
tok_ckpt = 'bert-base-uncased'
orig_tokenizer = AutoTokenizer.from_pretrained(tok_ckpt)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [36]:
orig_tokenizer

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [37]:
from torch.optim import AdamW

In [38]:
BroEncoder = Encoder(num_layers=6, num_heads=8, vocab_size=9000, 
                     embed_dim=256, k_dim=32, ctx_len=256, do=0.1, device=device)

In [39]:
BroEncoder

Encoder(
  (emb): Embed(
    (tok_layer): Embedding(9000, 256)
    (pos_layer): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (norm_do): Sequential(
      (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (layers): Sequential(
    (0): Encoder_Block(
      (MHA): Attention_Block(
        (heads): ModuleList(
          (0-7): 8 x Attention(
            (query): Linear(in_features=256, out_features=32, bias=True)
            (key): Linear(in_features=256, out_features=32, bias=True)
            (value): Linear(in_features=256, out_features=32, bias=True)
            (att_do): Dropout(p=0.1, inplace=False)
          )
        )
        (lin_do): Sequential(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): Dropout(p=0.1, inplace=False)
        )
      )
      (mha_blk_end_layernorm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
    

In [40]:
summary(BroEncoder)

Layer (type:depth-idx)                        Param #
Encoder                                       --
├─Embed: 1-1                                  --
│    └─Embedding: 2-1                         2,304,000
│    └─PositionalEncoding: 2-2                --
│    │    └─Dropout: 3-1                      --
│    └─Sequential: 2-3                        --
│    │    └─LayerNorm: 3-2                    512
│    │    └─Dropout: 3-3                      --
├─Sequential: 1-2                             --
│    └─Encoder_Block: 2-4                     --
│    │    └─Attention_Block: 3-4              263,168
│    │    └─LayerNorm: 3-5                    512
│    │    └─Sequential: 3-6                   525,568
│    │    └─LayerNorm: 3-7                    512
│    └─Encoder_Block: 2-5                     --
│    │    └─Attention_Block: 3-8              263,168
│    │    └─LayerNorm: 3-9                    512
│    │    └─Sequential: 3-10                  525,568
│    │    └─LayerNorm: 3-11      

##### imdb ds

In [41]:
ds_name_2 = 'stanfordnlp/imdb'
ds_imdb = load_dataset(ds_name_2)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [42]:
ds_imdb

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [43]:
tokenizer = AutoTokenizer.from_pretrained('./agnews_tokenizer')
tokenizer

BertTokenizerFast(name_or_path='./agnews_tokenizer', vocab_size=9000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [44]:
df_train = ds_imdb['train'].to_pandas()
df_train

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [46]:
df_test = ds_imdb['test'].shuffle(seed=42).select(range(1000)).to_pandas()

In [47]:
df_test['label'].value_counts()

label
0    512
1    488
Name: count, dtype: int64

In [48]:
df = pd.concat([df_train, df_test], axis=0)
df

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
995,I thought this movie was great! I saw this whe...,1
996,"Yes, it's a SBIF (So Bad It's Funny) classic. ...",0
997,"I am sorry folks, but I have to say I really c...",0
998,I suspect there are several cuts of this doco ...,1


In [53]:
df_train

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [54]:
df_train['label'].value_counts()

label
0    12500
1    12500
Name: count, dtype: int64

In [55]:
df_test['label'].value_counts()

label
0    512
1    488
Name: count, dtype: int64

In [59]:
class Bro_Classification_Model(nn.Module):
    def __init__(self, max_length, vocab_size, device):
        super().__init__()
        self.BroEncoder = Encoder(num_layers=6, num_heads=8, vocab_size=vocab_size,
                                  embed_dim=256, k_dim=32, ctx_len=256, do=0.1, device=device)
        self.reduce_dim = nn.Sequential(
            nn.Linear(256, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(64, 1)
        )
        self.gen_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.05),
            )
        self.classifier = nn.Linear(64, 2)
       

    def forward(self, input_ids, attention_mask):
        x = self.BroEncoder(input_ids, attention_mask)
        #x = x.mean(dim=1)
        x = self.reduce_dim(x)
        #print(x.shape)
        x = self.gen_layer(x)
        #print(x.shape)
        #x = x + y
        x = self.classifier(x)
        return x

In [60]:
vocab_size = tokenizer.vocab_size
vocab_size

9000

In [61]:
bro_model = Bro_Classification_Model(256, vocab_size, device)

In [62]:
summary(bro_model)

Layer (type:depth-idx)                             Param #
Bro_Classification_Model                           --
├─Encoder: 1-1                                     --
│    └─Embed: 2-1                                  --
│    │    └─Embedding: 3-1                         2,304,000
│    │    └─PositionalEncoding: 3-2                --
│    │    └─Sequential: 3-3                        512
│    └─Sequential: 2-2                             --
│    │    └─Encoder_Block: 3-4                     789,760
│    │    └─Encoder_Block: 3-5                     789,760
│    │    └─Encoder_Block: 3-6                     789,760
│    │    └─Encoder_Block: 3-7                     789,760
│    │    └─Encoder_Block: 3-8                     789,760
│    │    └─Encoder_Block: 3-9                     789,760
├─Sequential: 1-2                                  --
│    └─Linear: 2-3                                 16,448
│    └─LayerNorm: 2-4                              128
│    └─GELU: 2-5                  

In [63]:
bro_model

Bro_Classification_Model(
  (BroEncoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(9000, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_en

In [64]:
t = torch.randint(0, 100, (2,256))
t

tensor([[81, 77, 85, 52, 35, 57, 25, 47, 70, 45, 58, 22, 65,  5, 60,  4, 11, 76,
         47,  2, 52, 19, 78, 48, 59,  3, 69, 57,  0, 30,  7, 40, 78, 42, 58, 12,
         77, 28, 26, 47, 16, 33, 92, 55, 49, 94, 63,  4, 11, 31, 19, 45, 76, 26,
         26, 87, 62, 82, 56, 80,  0,  7, 64,  4, 11, 69, 82, 65, 80, 64, 15, 16,
          2, 89,  7, 36, 73, 19, 81, 42, 36, 59, 79, 16, 84, 63, 25,  9, 57, 88,
         25, 57, 88, 22, 63, 21, 48, 61, 17, 47, 32, 75, 63, 66, 87, 70, 72, 55,
         48, 60, 21,  7, 25,  6, 93, 57, 18, 36, 63, 73,  5, 26, 16,  1, 16, 64,
         96, 33, 86, 88, 60, 93, 84, 92, 31, 80, 46, 92, 37,  6, 52, 60, 20, 27,
         98, 58, 96,  5, 26, 85,  7,  1, 24,  7, 97, 31, 29, 74, 32, 59, 20, 37,
         27, 77, 95,  9, 78,  4, 92,  0, 75, 97, 90, 28,  8, 98, 19, 39,  7, 86,
         36, 36, 58, 89, 67, 87, 82, 26, 49, 29,  3, 13, 41, 88, 38, 21, 23, 75,
         13, 59, 27, 86, 89, 19, 51, 19,  1, 57, 80, 95, 84, 27, 24, 39, 82, 47,
         98, 72,  7, 53, 96,

In [65]:
am = torch.randint(1, 2, (2,256))
am

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [66]:
t.shape

torch.Size([2, 256])

In [67]:
t, am= t.to(device), am.to(device)

In [68]:
bro_model.to(device)

Bro_Classification_Model(
  (BroEncoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(9000, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_en

In [69]:
out = bro_model(t, am)
out, out.shape

(tensor([[-0.1954,  0.5039],
         [ 0.5378,  0.0986]], device='cuda:0', grad_fn=<AddmmBackward0>),
 torch.Size([2, 2]))

In [70]:
tokenizer = AutoTokenizer.from_pretrained('./agnews_tokenizer/')
tokenizer

BertTokenizerFast(name_or_path='./agnews_tokenizer/', vocab_size=9000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [71]:
df_train

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [72]:
df_train['label'].value_counts()

label
0    12500
1    12500
Name: count, dtype: int64

In [73]:
df_test['label'].value_counts()

label
0    512
1    488
Name: count, dtype: int64

In [74]:
class IMDB_DS(Dataset):
    def __init__(self, df, max_len):
        self.data = df['text']
        self.label = df['label']
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data.iloc[idx]
        y = torch.tensor(self.label.iloc[idx])
        tokenized = tokenizer(x, truncation=True, max_length=self.max_len, padding='max_length', return_tensors='pt')
        input_ids = tokenized.input_ids.squeeze()
        attention_mask = tokenized.attention_mask.squeeze()
        return {'input_ids': input_ids, 'attention_mask': attention_mask}, y

In [75]:
train_ds = IMDB_DS(df_train, 256)
test_ds = IMDB_DS(df_test, 256)

In [76]:
train_ds[0]

({'input_ids': tensor([   2,   42, 7870,  104,   42,  285, 3341,  839,   15, 7698,  241, 1428,
          1471, 2135, 1282,  115,  434,  102, 5938,  182, 5583,  104,  207,  591,
           207,  238,  343, 1392,  108, 7942,   95,   16,   42, 1653, 5820,  182,
           166,  343,  207,  238, 5374,  217,   54,   16,   52,   16, 1406,   68,
           838,  207, 1025, 4288,  111, 1176,  397,  890,   14, 1038, 2676,   65,
          1227,   34, 4203,  115, 8245, 5635,    6, 3692,    6,   42, 4330,  566,
           111, 1929,  397,  135, 7744, 1664,   16,    1,  333,   17,    1,    1,
           333,   17,    1,  102, 3787,  174, 1682,  104, 1545,   34, 2658, 6730,
          8366, 7092, 2648, 6060,   70,  504, 2250,  111, 6823, 5544, 1541,  400,
           488, 1561,   16,  108, 6642, 1541, 2250,  111, 2301,  951, 4958,   68,
           111, 1739,  693, 8031,  115, 6699,  427,  131,  938,  102, 3053, 3724,
            65, 3460,  488, 3564, 1788, 2777, 1897,  162,  102, 5408,  453,  132,
   

In [77]:
test_ds[0]

({'input_ids': tensor([   2,    1,  333,   17,    1,    1,  333,   17,    1,  591,   42, 4633,
           155, 1733,  816,  104,  169, 7870,  104,   34, 2019,  187,  263, 2039,
            14,   42, 3460,   42,  238,  108,  135,  117, 3074,  109, 2731, 3433,
          3894,  132,  115, 3121, 7980,  146, 4989,   65, 6112,   71,  238,  108,
           207,   14,  492,  938,  592,  312, 3759,   31,    1,  333,   17,    1,
             1,  333,   17,    1, 3065, 4602,   14, 6034,   14,   42, 1356, 1553,
           182,  397, 3894,  238,  488,   34, 2019,  187,  845, 3795, 4432, 1901,
           733,  263, 2039,   16,   42, 3312, 5488,  109,  132, 6190,   10,   53,
          1810, 2464,  779,  264,  102, 3339, 2344,   16,  436,   73,  824, 2175,
            65,   14, 6436,   70,  132,  777, 2944, 5844,   14,  135, 4705,  219,
          1897,   34, 6664, 7310, 1091,  576,  132,  224, 5283, 2134, 5943, 3339,
             5,  436,   73,  824, 3421,   14,  135, 1227, 4945,  132, 1564,  161,
   

In [78]:
train_dl = DataLoader(train_ds, shuffle=True, batch_size=32, num_workers=os.cpu_count())
test_dl = DataLoader(test_ds, shuffle=False, batch_size=32, num_workers=os.cpu_count())

In [79]:
len(train_dl.dataset), len(test_dl.dataset)

(25000, 1000)

In [80]:
i, l = next(iter(train_dl))
i.keys, i['input_ids'].shape, i['attention_mask'].shape, l.shape

(<function dict.keys>,
 torch.Size([32, 256]),
 torch.Size([32, 256]),
 torch.Size([32]))

In [81]:
i = {k: v.to(device) for k, v in i.items()}
i

{'input_ids': tensor([[   2,   34, 1266,  ...,    0,    0,    0],
         [   2,  397,  174,  ...,    0,    0,    0],
         [   2,  397,  174,  ...,    0,    0,    0],
         ...,
         [   2,   42,  310,  ..., 2273,   14,    3],
         [   2, 7911, 7734,  ...,    0,    0,    0],
         [   2,  102,  445,  ..., 4267,   68,    3]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1]], device='cuda:0')}

In [82]:
out = bro_model(**i)
out.shape

torch.Size([32, 2])

In [83]:
torch.cuda.empty_cache()

In [84]:
def test(model, dl, device, vocab_size):
    model.eval()
    fn_loss = nn.CrossEntropyLoss()
    model = model.to(device)

    loss_epoch = 0
    accuracy = 0
    for inputs in dl:
        data = inputs[0]
        label = inputs[1]
        batch_size = data['input_ids'].shape[0]
        ctx_size = data['input_ids'].shape[1]
        data = {i: k.to(device) for i, k in data.items()}
        label = label.to(device)
        with torch.no_grad():
            out = model(**data)
        #out = out.view(batch_size * ctx_size, vocab_size)
        #label = label.view(batch_size * ctx_size)
        loss = fn_loss(out, label)
        loss_epoch = loss_epoch + (loss.item() * batch_size)
        accuracy += (torch.argmax(out, dim=1) == label).sum().item()
    average_loss = loss_epoch/len(dl.dataset)
    average_acc = accuracy/len(dl.dataset)
    print(f'Average Test loss: {average_loss}')
    print(f'Average Test Accuracy: {average_acc}')
    return average_loss, average_acc

In [90]:
def train_model(num_epochs, lrate, model, dl, test_dl, device, vocab_size):
    epochs = num_epochs
    lr = lrate
    fn_loss = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=lr)
    grad_accum_steps = 1 #args.grad_accum_steps # 4 - previous static value
    model = model.to(device)
    base_test_acc = -torch.inf

    for i in range(epochs):
        loss_epoch = 0
        n_step = 0
        grad_accum_counter = 1
        accuracy = 0
        model.train()
        

        for inputs in dl:
            data = inputs[0]
            label = inputs[1]
            batch_size = data['input_ids'].shape[0]
            ctx_size = data['input_ids'].shape[1]
            data = {i: k.to(device) for i, k in data.items()}
            label = label.to(device)
            out = model(**data)
            #out = out.view(batch_size * ctx_size, vocab_size)
            #label = label.view(batch_size * ctx_size)
            loss = fn_loss(out, label)
            loss = loss / grad_accum_steps
            loss.backward()
            if grad_accum_counter == grad_accum_steps:
                optimizer.step()
                optimizer.zero_grad()
                grad_accum_counter = 0
            loss_epoch = loss_epoch + (loss.item() * batch_size * grad_accum_steps)
            accuracy += (torch.argmax(out, dim=1) == label).sum().item()
            if n_step % 200 == 0:
                print(f'Step: {n_step}, Loss: {loss.item()}')
            grad_accum_counter += 1
            n_step += 1

        average_loss = loss_epoch/len(dl.dataset)
        average_acc = accuracy/len(dl.dataset)
        print(f'Epoch: {i} -- Average training Loss: {average_loss}')
        print(f'Epoch: {i} -- Average training Accuracy: {average_acc}')
        avg_test_loss, avg_test_acc = test(model, test_dl, device, vocab_size)
        if avg_test_acc > base_test_acc:
            base_test_acc = avg_test_acc
            print("saving model")
            torch.save({
                'epoch': i,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_test_loss,
                'accuracy': avg_test_acc,
                'device': device
                }, './imdb_class_model_raw.pt')

    return model, optimizer, average_loss, average_acc

In [91]:
torch.cuda.empty_cache()

In [92]:
bro_model

Bro_Classification_Model(
  (BroEncoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(9000, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_en

In [93]:
model, optimizer, average_loss, average_acc = train_model(num_epochs=10, lrate=0.0003,
                                                          model=bro_model, dl=train_dl,
                                                          test_dl=test_dl, device=device,
                                                          vocab_size=2)

Step: 0, Loss: 0.7150650024414062
Step: 200, Loss: 0.7297382950782776
Step: 400, Loss: 0.4983130693435669
Step: 600, Loss: 0.398889422416687
Epoch: 0 -- Average training Loss: 0.5867479636669158
Epoch: 0 -- Average training Accuracy: 0.65988
Average Test loss: 0.9104096698760986
Average Test Accuracy: 0.6
saving model
Step: 0, Loss: 0.6119054555892944
Step: 200, Loss: 0.32069432735443115
Step: 400, Loss: 0.41750872135162354
Step: 600, Loss: 0.4645041525363922
Epoch: 1 -- Average training Loss: 0.4288161251115799
Epoch: 1 -- Average training Accuracy: 0.802
Average Test loss: 0.53527674305439
Average Test Accuracy: 0.777
saving model
Step: 0, Loss: 0.2924008071422577
Step: 200, Loss: 0.5643203854560852
Step: 400, Loss: 0.44485631585121155
Step: 600, Loss: 0.26949355006217957
Epoch: 2 -- Average training Loss: 0.3766995921421051
Epoch: 2 -- Average training Accuracy: 0.83236
Average Test loss: 1.6894965114593505
Average Test Accuracy: 0.51
Step: 0, Loss: 0.1998034417629242
Step: 200, Los